<a href="https://colab.research.google.com/github/josuemarroquinj/enterprise-risk-stress-testing-santander/blob/main/notebooks/01_financial_risk_dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 01 — Financial & Risk Dashboard

## Banco Santander 2025 Baseline Analysis

This notebook builds the financial and risk baseline for Banco Santander using publicly available information from the 2025 Annual Report.

### Objectives

- Structure the main reported financial indicators.
- Analyze profitability, efficiency, credit quality and solvency.
- Compare 2023–2025 trends.
- Establish the baseline for subsequent stress-testing exercises.
- Maintain clear traceability between reported data and derived analytical metrics.

### Analytical principle

The project distinguishes four types of information:

- **REPORTED** — directly reported by Banco Santander.
- **DERIVED** — calculated from reported information.
- **ASSUMPTION** — analytical assumption introduced by the researcher.
- **MODEL_OUTPUT** — result produced by the analytical model.

In [13]:
# ============================================================
# PROJECT ENVIRONMENT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Repository configuration
REPO_URL = "https://github.com/josuemarroquinj/enterprise-risk-stress-testing-santander.git"

PROJECT_ROOT = Path(
    "/content/enterprise-risk-stress-testing-santander"
)

# Clone repository only if it does not already exist
if not PROJECT_ROOT.exists():
    !git clone {REPO_URL}
else:
    print("Repository already available.")

# Project paths
DATA_DIR = PROJECT_ROOT / "data"
RAW_DATA = DATA_DIR / "raw"
PROCESSED_DATA = DATA_DIR / "processed"
ASSUMPTIONS_DATA = DATA_DIR / "assumptions"

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
FIGURES_DIR = OUTPUTS_DIR / "figures"
TABLES_DIR = OUTPUTS_DIR / "tables"

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data: {PROCESSED_DATA}")
print("Environment successfully initialized.")

Repository already available.
Project root: /content/enterprise-risk-stress-testing-santander
Processed data: /content/enterprise-risk-stress-testing-santander/data/processed
Environment successfully initialized.


In [17]:
baseline_records = [
    # --------------------------------------------------------
    # BALANCE SHEET
    # --------------------------------------------------------
    [2023, "balance_sheet", "total_assets", 1797062, "EUR_m", "REPORTED"],
    [2024, "balance_sheet", "total_assets", 1837081, "EUR_m", "REPORTED"],
    [2025, "balance_sheet", "total_assets", 1867515, "EUR_m", "REPORTED"],

    [2023, "balance_sheet", "customer_loans", 1036349, "EUR_m", "REPORTED"],
    [2024, "balance_sheet", "customer_loans", 1054069, "EUR_m", "REPORTED"],
    [2025, "balance_sheet", "customer_loans", 1037288, "EUR_m", "REPORTED"],

    [2023, "balance_sheet", "customer_deposits", 1047169, "EUR_m", "REPORTED"],
    [2024, "balance_sheet", "customer_deposits", 1055936, "EUR_m", "REPORTED"],
    [2025, "balance_sheet", "customer_deposits", 1041200, "EUR_m", "REPORTED"],

    [2023, "balance_sheet", "customer_resources", 1306942, "EUR_m", "REPORTED"],
    [2024, "balance_sheet", "customer_resources", 1348422, "EUR_m", "REPORTED"],
    [2025, "balance_sheet", "customer_resources", 1363160, "EUR_m", "REPORTED"],

    [2023, "balance_sheet", "equity", 104241, "EUR_m", "REPORTED"],
    [2024, "balance_sheet", "equity", 107327, "EUR_m", "REPORTED"],
    [2025, "balance_sheet", "equity", 112748, "EUR_m", "REPORTED"],

    # --------------------------------------------------------
    # INCOME STATEMENT
    # --------------------------------------------------------
    [2023, "income_statement", "net_interest_income", 40650, "EUR_m", "REPORTED"],
    [2024, "income_statement", "net_interest_income", 43787, "EUR_m", "REPORTED"],
    [2025, "income_statement", "net_interest_income", 42348, "EUR_m", "REPORTED"],

    [2023, "income_statement", "gross_income", 54251, "EUR_m", "REPORTED"],
    [2024, "income_statement", "gross_income", 58380, "EUR_m", "REPORTED"],
    [2025, "income_statement", "gross_income", 58670, "EUR_m", "REPORTED"],

    [2023, "income_statement", "net_operating_income", 29619, "EUR_m", "REPORTED"],
    [2024, "income_statement", "net_operating_income", 33231, "EUR_m", "REPORTED"],
    [2025, "income_statement", "net_operating_income", 33959, "EUR_m", "REPORTED"],

    [2023, "income_statement", "profit_before_tax", 15005, "EUR_m", "REPORTED"],
    [2024, "income_statement", "profit_before_tax", 17347, "EUR_m", "REPORTED"],
    [2025, "income_statement", "profit_before_tax", 18681, "EUR_m", "REPORTED"],

    [2023, "income_statement", "attributable_profit", 11076, "EUR_m", "REPORTED"],
    [2024, "income_statement", "attributable_profit", 12574, "EUR_m", "REPORTED"],
    [2025, "income_statement", "attributable_profit", 14101, "EUR_m", "REPORTED"],

    # --------------------------------------------------------
    # PROFITABILITY
    # --------------------------------------------------------
    [2023, "profitability", "roe", 11.9, "percent", "REPORTED"],
    [2024, "profitability", "roe", 13.0, "percent", "REPORTED"],
    [2025, "profitability", "roe", 13.9, "percent", "REPORTED"],

    [2023, "profitability", "rote", 15.1, "percent", "REPORTED"],
    [2024, "profitability", "rote", 16.3, "percent", "REPORTED"],
    [2025, "profitability", "rote", 17.1, "percent", "REPORTED"],

    [2023, "profitability", "rote_post_at1", 14.4, "percent", "REPORTED"],
    [2024, "profitability", "rote_post_at1", 15.5, "percent", "REPORTED"],
    [2025, "profitability", "rote_post_at1", 16.3, "percent", "REPORTED"],

    [2023, "profitability", "roa", 0.69, "percent", "REPORTED"],
    [2024, "profitability", "roa", 0.76, "percent", "REPORTED"],
    [2025, "profitability", "roa", 0.84, "percent", "REPORTED"],

    [2023, "profitability", "rorwa", 1.96, "percent", "REPORTED"],
    [2024, "profitability", "rorwa", 2.18, "percent", "REPORTED"],
    [2025, "profitability", "rorwa", 2.44, "percent", "REPORTED"],

    [2023, "profitability", "efficiency_ratio", 44.1, "percent", "REPORTED"],
    [2024, "profitability", "efficiency_ratio", 41.8, "percent", "REPORTED"],
    [2025, "profitability", "efficiency_ratio", 41.2, "percent", "REPORTED"],

    # --------------------------------------------------------
    # CREDIT QUALITY
    # --------------------------------------------------------
    [2023, "credit_risk", "cost_of_risk", 1.18, "percent", "REPORTED"],
    [2024, "credit_risk", "cost_of_risk", 1.15, "percent", "REPORTED"],
    [2025, "credit_risk", "cost_of_risk", 1.15, "percent", "REPORTED"],

    [2023, "credit_risk", "npl_ratio", 3.14, "percent", "REPORTED"],
    [2024, "credit_risk", "npl_ratio", 3.05, "percent", "REPORTED"],
    [2025, "credit_risk", "npl_ratio", 2.91, "percent", "REPORTED"],

    [2023, "credit_risk", "npl_coverage", 66, "percent", "REPORTED"],
    [2024, "credit_risk", "npl_coverage", 65, "percent", "REPORTED"],
    [2025, "credit_risk", "npl_coverage", 66, "percent", "REPORTED"],

    # --------------------------------------------------------
    # CAPITAL
    # --------------------------------------------------------
    [2024, "capital", "cet1_capital", 79800, "EUR_m", "REPORTED"],
    [2025, "capital", "cet1_capital", 84739, "EUR_m", "REPORTED"],

    [2024, "capital", "tier1_capital", 90170, "EUR_m", "REPORTED"],
    [2025, "capital", "tier1_capital", 94385, "EUR_m", "REPORTED"],

    [2024, "capital", "total_capital", 108589, "EUR_m", "REPORTED"],
    [2025, "capital", "total_capital", 111845, "EUR_m", "REPORTED"],

    [2024, "capital", "rwa", 624503, "EUR_m", "REPORTED"],
    [2025, "capital", "rwa", 629430, "EUR_m", "REPORTED"],

    [2024, "capital", "cet1_ratio", 12.8, "percent", "REPORTED"],
    [2025, "capital", "cet1_ratio", 13.46, "percent", "REPORTED"],

    [2024, "capital", "total_capital_ratio", 17.4, "percent", "REPORTED"],
    [2025, "capital", "total_capital_ratio", 17.8, "percent", "REPORTED"],

    [2024, "capital", "leverage_ratio", 4.78, "percent", "REPORTED"],
    [2025, "capital", "leverage_ratio", 4.90, "percent", "REPORTED"],
]

financial_baseline = pd.DataFrame(
    baseline_records,
    columns=[
        "year",
        "category",
        "metric",
        "value",
        "unit",
        "data_type"
    ]
)

financial_baseline.head(10)



,year,category,metric,value,unit,data_type
0,2023,balance_sheet,total_assets,"1,797,062.00",EUR_m,REPORTED
1,2024,balance_sheet,total_assets,"1,837,081.00",EUR_m,REPORTED
2,2025,balance_sheet,total_assets,"1,867,515.00",EUR_m,REPORTED
3,2023,balance_sheet,customer_loans,"1,036,349.00",EUR_m,REPORTED
4,2024,balance_sheet,customer_loans,"1,054,069.00",EUR_m,REPORTED
5,2025,balance_sheet,customer_loans,"1,037,288.00",EUR_m,REPORTED
6,2023,balance_sheet,customer_deposits,"1,047,169.00",EUR_m,REPORTED
7,2024,balance_sheet,customer_deposits,"1,055,936.00",EUR_m,REPORTED
8,2025,balance_sheet,customer_deposits,"1,041,200.00",EUR_m,REPORTED
9,2023,balance_sheet,customer_resources,"1,306,942.00",EUR_m,REPORTED


## 2. Data Loading

The financial baseline dataset is loaded from the processed data directory.

This dataset contains the standardized financial and risk indicators used throughout the analysis.

In [14]:
financial_data = pd.read_csv(
    PROCESSED_DATA / "financial_baseline_2023_2025.csv"
)

financial_data.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/enterprise-risk-stress-testing-santander/data/processed/financial_baseline_2023_2025.csv'

In [ ]:
balance_data = pd.DataFrame({
    "year": [2023, 2024, 2025],
    "total_assets_eur_m": [1797062, 1837081, 1867515],
    "customer_loans_eur_m": [1036349, 1054069, 1037288],
    "customer_deposits_eur_m": [1047169, 1055936, 1041200],
    "customer_resources_eur_m": [1306942, 1348422, 1363160],
    "equity_eur_m": [104241, 107327, 112748]
})

balance_data["data_type"] = "REPORTED"

balance_data

In [ ]:
income_data = pd.DataFrame({
    "year": [2023, 2024, 2025],
    "net_interest_income_eur_m": [40650, 43787, 42348],
    "gross_income_eur_m": [54251, 58380, 58670],
    "net_operating_income_eur_m": [29619, 33231, 33959],
    "profit_before_tax_eur_m": [15005, 17347, 18681],
    "attributable_profit_eur_m": [11076, 12574, 14101]
})

income_data["data_type"] = "REPORTED"

income_data

In [ ]:
profitability_data = pd.DataFrame({
    "year": [2023, 2024, 2025],
    "roe_pct": [11.9, 13.0, 13.9],
    "rote_pct": [15.1, 16.3, 17.1],
    "rote_post_at1_pct": [14.4, 15.5, 16.3],
    "roa_pct": [0.69, 0.76, 0.84],
    "rorwa_pct": [1.96, 2.18, 2.44],
    "efficiency_ratio_pct": [44.1, 41.8, 41.2]
})

profitability_data["data_type"] = "REPORTED"

profitability_data

In [ ]:
credit_quality = pd.DataFrame({
    "year": [2023, 2024, 2025],
    "cost_of_risk_pct": [1.18, 1.15, 1.15],
    "npl_ratio_pct": [3.14, 3.05, 2.91],
    "npl_coverage_pct": [66, 65, 66]
})

credit_quality["data_type"] = "REPORTED"

credit_quality

In [ ]:
capital_data = pd.DataFrame({
    "year": [2024, 2025],
    "cet1_capital_eur_m": [79800, 84739],
    "tier1_capital_eur_m": [90170, 94385],
    "total_capital_eur_m": [108589, 111845],
    "rwa_eur_m": [624503, 629430],
    "cet1_ratio_pct": [12.8, 13.5],
    "total_capital_ratio_pct": [17.4, 17.8],
    "leverage_ratio_pct": [4.78, 4.90]
})

capital_data["data_type"] = "REPORTED"

capital_data

In [ ]:
cet1_ratio_exact = 13.46
cet1_requirement = 9.84

cet1_buffer_pp = cet1_ratio_exact - cet1_requirement
cet1_buffer_bp = cet1_buffer_pp * 100

print(f"CET1 ratio:        {cet1_ratio_exact:.2f}%")
print(f"CET1 requirement:  {cet1_requirement:.2f}%")
print(f"Capital buffer:    {cet1_buffer_pp:.2f} percentage points")
print(f"Capital buffer:    {cet1_buffer_bp:.0f} basis points")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(
    profitability_data["year"],
    profitability_data["rote_pct"],
    marker="o",
    linewidth=2,
    label="RoTE"
)

ax.plot(
    profitability_data["year"],
    profitability_data["efficiency_ratio_pct"],
    marker="o",
    linewidth=2,
    label="Efficiency Ratio"
)

ax.set_title("Banco Santander — Profitability and Efficiency")
ax.set_xlabel("Year")
ax.set_ylabel("%")
ax.set_xticks(profitability_data["year"])
ax.legend()
ax.grid(alpha=0.3)

plt.show()

In [ ]:
income_analysis = income_data.copy()

income_analysis["attributable_profit_growth_pct"] = (
    income_analysis["attributable_profit_eur_m"]
    .pct_change()
    .mul(100)
)

income_analysis